# 14. LG-CoTrain full grid with default hyperparameters

Runs the 120-cell LG-CoTrain grid (10 events x 4 budgets x 3 seeds) on the BERTweet backbone with the LG-CoTrain reference paper's default hyperparameters. Produces one `metrics.json` per cell.

## What this notebook does

| Stage | What | Approx. time |
|---|---|---|
| 0 | Smoke test: 3 cells (kaikoura, budget=5, seeds 1/2/3) to verify multi-GPU dispatch works on the host | ~7-10 min |
| 1 | Full grid: 10 events x 4 budgets x 3 seeds = 120 cells, parallel across all available GPUs | ~14 hours on 2 consumer GPUs; faster on more/larger GPUs |
| 2 | Sanity check: count produced `metrics.json` files and print per-budget summary | ~1 min (no GPU) |

## Settings

- Backbone: `vinai/bertweet-base`
- Pseudo-label source: `gpt-4o` (read from `data/pseudo-labelled/gpt-4o/`)
- Hyperparameters: `lr=2e-5`, `batch_size=32`, `cotrain_epochs=10`, `finetune_patience=5`, `weight_gen_epochs=7`, `weight_decay=0.01`, `warmup_ratio=0.1`
- Output tree: `results/bertweet/gpt-4o/defaults/wge7/`

## Notes before running

- **Disable system sleep before launching Stage 1.** A 14-hour run will fail if the OS suspends Python partway through.
- **Stage 1 ties up the kernel for hours.** If Jupyter disconnects, the underlying process keeps running and writes results to disk; you can monitor by counting `metrics.json` files under the output tree.
- **Resumable**: re-running Stage 1 skips cells whose `metrics.json` already exists.


## Setup


In [ ]:
import sys
import subprocess
import time
import json
import shutil
import re
from pathlib import Path

# Locate the repo root by walking up from the notebook's working directory
def _find_repo_root(marker: str = "lg_cotrain") -> Path:
    for candidate in [Path().resolve()] + list(Path().resolve().parents):
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Cannot find repo root: no ancestor directory contains '{marker}/'")

repo_root = _find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"Repo root: {repo_root}")
print(f"Python:    {sys.version.split()[0]}")


In [ ]:
# Check that both GPUs are visible to PyTorch and have enough free VRAM
try:
    import torch
    print(f"torch version:        {torch.__version__}")
    print(f"CUDA available:       {torch.cuda.is_available()}")
    if not torch.cuda.is_available():
        raise RuntimeError("No CUDA GPU detected. The notebook requires at least 1 GPU.")
    n_gpus = torch.cuda.device_count()
    print(f"CUDA device count:    {n_gpus}")
    if n_gpus < 2:
        print()
        print("WARNING: only 1 GPU visible. Stage 1 will run sequentially and take ~27 hours")
        print("instead of ~14 hours with 2-GPU parallelism.")
    for i in range(n_gpus):
        props = torch.cuda.get_device_properties(i)
        free_gb, total_gb = (x / 1024**3 for x in torch.cuda.mem_get_info(i))
        print(f"  cuda:{i}  {props.name}  total={total_gb:.1f} GB  free={free_gb:.1f} GB")
except ImportError:
    raise RuntimeError("torch not installed. Activate your training environment first.")

try:
    import transformers
    print(f"transformers version: {transformers.__version__}")
except ImportError:
    raise RuntimeError("transformers not installed. Activate your training environment first.")

print()
print("REMINDER: Disable Windows sleep before launching the 14-hour Stage 1 cell.")
print("  Settings > System > Power > Screen and sleep > set sleep to Never (when plugged in)")


## Stage 0: smoke test (3 cells, ~7-8 min)

Runs 3 BERTweet cells (`kaikoura_earthquake_2016`, `budget=5`, seeds 1/2/3) with `--num-gpus 2` to verify:

1. The parallel runner works on Windows (no spawn-mode pickling failures, no CUDA conflicts)
2. Both GPUs receive work (the 3-cell queue means at least one GPU has to handle 2 cells while the other handles 1, exercising work-stealing)
3. BERTweet loads and trains end-to-end through Phase 1, Phase 2, and Phase 3
4. The metrics are reasonable (sanity bounds 0.3-0.95)

Output goes to `results-bertweet-smoke/` (a separate folder) so the smoke test results don't pollute the canonical `results-bertweet/` tree. After Stage 0 passes, the next cell deletes `results-bertweet-smoke/` so Stage 1 starts clean.

If Stage 0 fails, do NOT proceed to Stage 1. The most common Windows-specific issue is `parallel.py` failing to dispatch across both GPUs. Fall back to `--num-gpus 1` and accept the longer wall clock if needed.


In [ ]:
# Stage 0: smoke test
SMOKE_OUT = Path(repo_root) / "results-bertweet-smoke" / "gpt-4o" / "optuna-stop" / "baseline"
SMOKE_OUT.mkdir(parents=True, exist_ok=True)

# Detect how many GPUs are visible and use the same number for the smoke test.
import torch
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 1
print(f"Detected {N_GPUS} GPUs. Using --num-gpus {N_GPUS} for the smoke test.")
print()

# Smoke test scope: 1 event x 3 budgets x 3 seeds = 9 cells.
# This is enough to exercise work-stealing on any GPU count up to 9 (e.g. on
# 8 GPUs, 8 cells start immediately and the 9th waits for the first GPU to
# finish, exercising the dispatcher's "grab next from queue" logic).
# kaikoura_earthquake_2016 is the smallest event so each cell finishes fast.
SMOKE_EVENTS = ["kaikoura_earthquake_2016"]
SMOKE_BUDGETS = [5, 10, 25]
SMOKE_SEEDS = [1, 2, 3]
EXPECTED_CELLS = len(SMOKE_EVENTS) * len(SMOKE_BUDGETS) * len(SMOKE_SEEDS)

cmd = [
    sys.executable, "-m", "lg_cotrain.run_experiment",
    "--events", *SMOKE_EVENTS,
    "--budgets", *(str(b) for b in SMOKE_BUDGETS),
    "--seed-sets", *(str(s) for s in SMOKE_SEEDS),
    "--num-gpus", str(N_GPUS),
    "--model-name", "vinai/bertweet-base",
    "--pseudo-label-source", "gpt-4o",
    "--data-root", str(Path(repo_root) / "data"),
    "--output-folder", str(SMOKE_OUT),
]
print("Running smoke test:")
print("  " + " ".join(cmd))
print()
print(f"Expected: {EXPECTED_CELLS} cells, ~2-3 min wall clock if N_GPUS >= {EXPECTED_CELLS - 1}")
print("=" * 70)

start = time.time()
proc = subprocess.run(cmd, cwd=str(repo_root))
elapsed = time.time() - start

print("=" * 70)
print(f"Wall clock: {elapsed:.0f} seconds = {elapsed/60:.1f} minutes")
if proc.returncode != 0:
    raise RuntimeError(f"Smoke test failed with exit code {proc.returncode}")

# Find all metrics.json files across all cells produced by the smoke test
metrics_files = []
for event in SMOKE_EVENTS:
    event_dir = SMOKE_OUT / event
    if event_dir.exists():
        metrics_files.extend(sorted(event_dir.glob("*/metrics.json")))

print()
print(f"Metrics files found: {len(metrics_files)} (expected {EXPECTED_CELLS})")
assert len(metrics_files) == EXPECTED_CELLS, (
    f"Expected {EXPECTED_CELLS} cells, found {len(metrics_files)}. Smoke test FAILED."
)

# Sanity check the metrics
print()
print("Per-cell sanity check:")
for f in metrics_files:
    m = json.loads(f.read_text())
    event = f.parent.parent.name
    cell_name = f.parent.name  # e.g. "5_set1"
    f1 = m["test_macro_f1"]
    err = m["test_error_rate"]
    assert 0.3 <= f1 <= 0.95, f"Out of range: {f} -> {f1}"
    print(f"  {event[:25]:<25s} {cell_name:<10s} test_macro_f1={f1:.4f}  test_error_rate={err:.2f}%  OK")

# Multi-GPU dispatch check (timing-based primary, log-parsing secondary)
print()
print("Multi-GPU dispatch check:")
print("-" * 60)

# Method 1 (primary): timing-based check.
# If all EXPECTED_CELLS cells run in parallel on N_GPUS where N_GPUS >= EXPECTED_CELLS,
# total wall clock is roughly the time of one (slowest) cell. If they run sequentially
# on 1 GPU, total wall clock is roughly the SUM of per-cell times.
import datetime as _dt

def _cell_wall_clock_seconds(log_path):
    """Parse the first and last timestamps from an experiment.log file."""
    if not log_path.exists():
        return None
    with open(log_path) as fh:
        lines = fh.readlines()
    if len(lines) < 2:
        return None
    try:
        first = _dt.datetime.strptime(lines[0][:19], "%Y-%m-%d %H:%M:%S")
        last = _dt.datetime.strptime(lines[-1][:19], "%Y-%m-%d %H:%M:%S")
        return (last - first).total_seconds()
    except Exception:
        return None

per_cell_times = [
    _cell_wall_clock_seconds(f.parent / "experiment.log")
    for f in metrics_files
]
per_cell_times = [t for t in per_cell_times if t is not None]
if per_cell_times:
    avg_per_cell = sum(per_cell_times) / len(per_cell_times)
    max_per_cell = max(per_cell_times)
    sequential_estimate = sum(per_cell_times)
    speedup = sequential_estimate / elapsed if elapsed > 0 else 1.0
    print(f"  Cells: {len(per_cell_times)}  per-cell avg={avg_per_cell:.0f}s  per-cell max={max_per_cell:.0f}s")
    print(f"  Sequential estimate (sum of per-cell times):  {sequential_estimate:.0f}s")
    print(f"  Actual total wall clock:                       {elapsed:.0f}s")
    print(f"  Speedup vs sequential:                         {speedup:.2f}x")
    print()
    expected_speedup = min(N_GPUS, EXPECTED_CELLS)
    if N_GPUS == 1:
        print(f"  Single GPU detected: speedup of ~1.0x is expected.")
    elif speedup >= expected_speedup * 0.7:
        print(f"  Speedup is ~{speedup:.1f}x out of an ideal ~{expected_speedup}x. Multi-GPU dispatch is working well.")
    elif speedup >= 2.0:
        print(f"  Speedup is moderate ({speedup:.1f}x out of ideal {expected_speedup}x).")
        print(f"  Some parallelism but not full. Could be due to: GPU heterogeneity,")
        print(f"  startup overhead, or small cells finishing too fast for the dispatcher to refill.")
    else:
        print(f"  WARNING: speedup is low ({speedup:.1f}x). Multi-GPU dispatch may not be working.")
        print(f"  Investigate before launching Stage 1.")

# Method 2 (secondary): parse cuda:N from log files (only works if trainer logs it)
print()
gpus_used = {}
for f in metrics_files:
    log_path = f.parent / "experiment.log"
    if not log_path.exists():
        continue
    text = log_path.read_text()
    match = re.search(r"Using device:\s*cuda:(\d+)", text)
    if not match:
        match = re.search(r"cuda:(\d+)", text)
    if match:
        gpu_id = int(match.group(1))
        cell_label = f"{f.parent.parent.name}/{f.parent.name}"
        gpus_used[cell_label] = gpu_id

if gpus_used:
    print("Per-cell device assignment (from experiment.log):")
    for cell_label, gpu_id in sorted(gpus_used.items()):
        print(f"  {cell_label}: cuda:{gpu_id}")
    unique_gpus = set(gpus_used.values())
    print(f"  -> Used {len(unique_gpus)} unique GPU(s): cuda:{sorted(unique_gpus)}")
    if N_GPUS >= EXPECTED_CELLS - 1 and len(unique_gpus) >= EXPECTED_CELLS - 1:
        print(f"  -> Nearly all available GPUs received work. Excellent.")
    elif len(unique_gpus) >= 2:
        print(f"  -> Multiple GPUs used. Multi-GPU dispatch confirmed.")
    else:
        print(f"  -> WARNING: only one GPU used. Multi-GPU dispatch may have failed.")
else:
    print("Note: experiment.log files do not contain 'cuda:N' device info.")
    print("(Older trainer.py versions did not log this. Pull the latest branch to get it.)")
    print("Falling back to timing-based check above.")

print()
print("STAGE 0 SMOKE TEST COMPLETE.")
print()
print("If the speedup is close to the GPU count, proceed to the cleanup cell")
print("then Stage 1. If speedup is much lower than expected, investigate first.")


In [ ]:
# Stage 0 cleanup: delete the smoke test folder so Stage 1 starts clean.
# This is a separate cell so you can inspect the smoke test results before deletion if needed.
smoke_root_to_remove = Path(repo_root) / "results-bertweet-smoke"
if smoke_root_to_remove.exists():
    print(f"Deleting smoke test output: {smoke_root_to_remove}")
    shutil.rmtree(smoke_root_to_remove)
    print("Cleanup done. Stage 1 can now run from a clean state.")
else:
    print(f"Nothing to clean up at {smoke_root_to_remove}")


## Stage 1: full grid (120 cells, ~14 hours)

Runs all 10 events x 4 budgets x 3 seeds = 120 cells with `--num-gpus 2`. Output goes to the canonical `results-bertweet/` folder.

Events are launched in increasing train-split size order so the first half of the run completes the small events quickly (early sanity check) and the long tail of large events runs at the end.

| Order | Event | Train tweets |
|---|---|---|
| 1 | kaikoura_earthquake_2016 | 1,536 |
| 2 | canada_wildfires_2016 | 1,569 |
| 3 | cyclone_idai_2019 | 2,753 |
| 4 | hurricane_florence_2018 | 4,384 |
| 5 | hurricane_maria_2017 | 5,094 |
| 6 | california_wildfires_2018 | 5,163 |
| 7 | hurricane_dorian_2019 | 5,329 |
| 8 | kerala_floods_2018 | 5,588 |
| 9 | hurricane_harvey_2017 | 6,378 |
| 10 | hurricane_irma_2017 | 6,579 |

**Important reminders:**
- Disable Windows sleep before running this cell
- Don't use the kernel for anything else for ~14 hours
- If interrupted, just re-run the cell — the resume logic skips cells whose `metrics.json` already exists
- You can monitor progress from a separate notebook by counting `metrics.json` files in `results-bertweet/gpt-4o/optuna-stop/baseline/`


In [ ]:
# Stage 1: full grid (~14 hours sequential, less with multi-GPU)
FULL_OUT = Path(repo_root) / "results" / "bertweet" / "gpt-4o" / "optuna-stop" / "baseline"
FULL_OUT.mkdir(parents=True, exist_ok=True)

# Auto-detect GPU count, same as Stage 0
import torch
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 1
print(f"Detected {N_GPUS} GPUs. Using --num-gpus {N_GPUS} for the full grid.")
print()

# Events in increasing train-split size order
EVENTS_ORDERED = [
    "kaikoura_earthquake_2016",   # 1,536
    "canada_wildfires_2016",      # 1,569
    "cyclone_idai_2019",          # 2,753
    "hurricane_florence_2018",    # 4,384
    "hurricane_maria_2017",       # 5,094
    "california_wildfires_2018",  # 5,163
    "hurricane_dorian_2019",      # 5,329
    "kerala_floods_2018",         # 5,588
    "hurricane_harvey_2017",      # 6,378
    "hurricane_irma_2017",        # 6,579
]

cmd = [
    sys.executable, "-m", "lg_cotrain.run_experiment",
    "--events", *EVENTS_ORDERED,
    "--num-gpus", str(N_GPUS),
    "--model-name", "vinai/bertweet-base",
    "--pseudo-label-source", "gpt-4o",
    "--data-root", str(Path(repo_root) / "data"),
    "--output-folder", str(FULL_OUT),
]
print("Running full grid:")
print("  " + " ".join(cmd))
print()
print("Expected wall clock: ~14 hours (120 cells, 2-GPU parallel)")
print("=" * 70)

start = time.time()
proc = subprocess.run(cmd, cwd=str(repo_root))
elapsed = time.time() - start

print("=" * 70)
print(f"Wall clock: {elapsed:.0f} seconds = {elapsed/60:.1f} minutes = {elapsed/3600:.1f} hours")
if proc.returncode != 0:
    print(f"WARNING: subprocess returned exit code {proc.returncode}")
    print("Some cells may have failed. Check metrics.json count below to see how many succeeded.")

# Count metrics.json files written
all_metrics = sorted(FULL_OUT.glob("*/*/metrics.json"))
print()
print(f"Metrics files written: {len(all_metrics)} (expected 120)")
if len(all_metrics) < 120:
    print(f"  MISSING: {120 - len(all_metrics)} cells")
    print("  Re-run this cell to retry the missing cells (resume logic will skip the completed ones)")
else:
    print("STAGE 1 COMPLETE: all 120 cells produced metrics.json")


## Stage 2: sanity check and comparison

Two cells, both pure-Python (no GPU, no training). Take just seconds.

**Sanity check** verifies that all 120 cells have valid metrics in expected ranges and that both GPUs were actually used during the run (work-stealing dispatch should give a roughly 55/45 split between A6000 and 3090).

**Comparison** computes the per-budget mean macro_f1 for both bert-base (existing) and BERTweet (new), and the paired delta. If the deltas are small (within ±0.03), the BERTweet results are ready to replace the bert-base ones in Table 3 of the paper.


In [ ]:
# Stage 2a: sanity check the 120 BERTweet metrics
import math
from collections import defaultdict

FULL_OUT = Path(repo_root) / "results" / "bertweet" / "gpt-4o" / "optuna-stop" / "baseline"
all_files = sorted(FULL_OUT.glob("*/*/metrics.json"))
print(f"Total metrics.json files: {len(all_files)}")
assert len(all_files) == 120, f"Expected 120, found {len(all_files)}"

# Required keys
REQUIRED_KEYS = [
    "test_macro_f1", "test_error_rate", "test_ece",
    "dev_macro_f1", "dev_error_rate", "dev_ece",
    "lambda1_mean", "lambda2_mean",
]

# Aggregate by (event, budget) and check bounds
by_event_budget = defaultdict(list)
gpus_used = defaultdict(int)
problems = []
for f in all_files:
    m = json.loads(f.read_text())
    event = f.parent.parent.name
    cell_name = f.parent.name  # e.g. "5_set1"
    budget, _, seed = cell_name.partition("_")
    budget = int(budget)
    seed = int(seed.replace("set", ""))

    # Required keys
    for k in REQUIRED_KEYS:
        if k not in m:
            problems.append(f"{f}: missing key {k}")
    # NaN check
    if math.isnan(m.get("test_macro_f1", float("nan"))):
        problems.append(f"{f}: test_macro_f1 is NaN")
    # Bounds
    if not (0.2 <= m.get("test_macro_f1", -1) <= 0.95):
        problems.append(f"{f}: test_macro_f1 out of bounds: {m.get('test_macro_f1')}")

    by_event_budget[(event, budget)].append(m["test_macro_f1"])

    # Count which GPU it ran on. New trainer.py logs "Using device: cuda:N".
    # Old runs may not have this. Try the new pattern first, then a generic
    # cuda:N anywhere in the log.
    log_path = f.parent / "experiment.log"
    if log_path.exists():
        text = log_path.read_text()
        match = re.search(r"Using device:\s*cuda:(\d+)", text)
        if not match:
            match = re.search(r"cuda:(\d+)", text)
        if match:
            gpus_used[int(match.group(1))] += 1

if problems:
    print()
    print(f"FOUND {len(problems)} PROBLEMS:")
    for p in problems[:20]:
        print(f"  {p}")
    raise RuntimeError("Sanity check failed. Investigate before proceeding.")

print("All 120 cells passed sanity check (no NaN, all keys present, all in 0.2-0.95 range)")
print()

# Per-event summary
print("Per-event mean test_macro_f1 (averaged across 3 seeds per budget):")
import statistics
events_seen = sorted(set(e for e, _ in by_event_budget))
print(f"  {'Event':<35s} {'b=5':>8s} {'b=10':>8s} {'b=25':>8s} {'b=50':>8s}")
for event in events_seen:
    row = [event[:35]]
    for budget in [5, 10, 25, 50]:
        vals = by_event_budget.get((event, budget), [])
        if vals:
            row.append(f"{statistics.mean(vals):.4f}")
        else:
            row.append("---")
    print("  " + " ".join(f"{c:>8s}" if i > 0 else f"{c:<35s}" for i, c in enumerate(row)))

# GPU usage summary (only meaningful if logs contain device info)
print()
if gpus_used:
    print(f"GPU usage across 120 cells: {dict(sorted(gpus_used.items()))}")
    total = sum(gpus_used.values())
    for gpu_id in sorted(gpus_used):
        pct = gpus_used[gpu_id] / total * 100
        print(f"  cuda:{gpu_id}: {gpus_used[gpu_id]} cells ({pct:.0f}%)")
    if len(gpus_used) < 2:
        print("  WARNING: only one GPU was used. Multi-GPU dispatch may have failed silently.")
else:
    print("No GPU device info found in experiment.log files. Skipping GPU usage summary.")
    print("(If you re-ran with the trainer.py fix on this branch, future logs will have it.)")
